In [344]:
#import any necessary libraries
import pandas as pd
import random
import ipywidgets as widgets
from IPython.display import display
from IPython.display import Image as NotebookImage
from IPython.display import Image as HTMLImage
from IPython.display import HTML
import requests
import json

In [287]:
# read animal dataset
animals_df = pd.read_csv('data/Animal Characteristics Data.csv')

# read scenario file
from pathlib import Path
scenarios = json.loads(Path('data/scenarios.json').read_text())

In [366]:
#Select an animal (the game is started in here!!!)
def select_animal(change): 
    global animal
    name = text_input.value.strip().capitalize()
    animal = animals_df.loc[animals_df.animal == name]

    with output:
        output.clear_output(wait=True)
        if animal.empty:
            print("Animal not found. Try again!")
        else:
            print("You selected:", animal["animal"].iloc[0])
            #update to update the global animal and change the output?
            start_game(animal)

In [270]:
def display_stats():
    print(f"Animal: {animal["animal"].iloc[0]}")
    print(f"Health: {animal_hp}")
    print(f"Energy: {animal_nrg}")
    print(f"Score: {player_score}")
    print("")

In [368]:
#find the percentile of a stat in a datast
def remap_num(animal, col):
    return animals_df[col].rank(pct=True).loc[animal.index[0]] * 100
    
# calculate player stats based on animal selection
def set_stats(animal):
    global animal_hp, animal_nrg, player_score

    starting_stat = 30
    # variable that cuts the percentile to prevent being too OP
    factor = 4
    
    # health
    animal_hp = round(
        starting_stat
        + (remap_num(animal, "weight_kg") 
        + remap_num(animal, "lifespan_years")
          ) / factor
    )
    
    animal_nrg = round(
        starting_stat 
        + (remap_num(animal, "speed_kmh")
        + (4 * animal["can_swim"].iloc[0])
        + (2 * animal["can_fly"].iloc[0])
        + (2 *animal["has_wings"].iloc[0])
          )/ factor
    )
    return animal_hp, animal_nrg

In [377]:
# select a random scenario from json file
def get_event():
    global curr_event, choices
    curr_event = scenarios[random.choice(list(scenarios.keys()))]
    choices = curr_event["choices"]

#update player stats accordingly when a choice is made
def choice_selected(button, choice):
    global animal_hp, animal_nrg, player_score 
    animal_hp += choice["health"]
    animal_nrg += choice["energy"]
    player_score += choice["score"]

    if animal_hp <= 0 or animal_nrg <= 0:
        with output: 
            output.clear_output(wait=True)
            display_stats()
            print("You died!")
            #display picture of animal but change the style so you know somehting happened
            display(HTML(f'<img src="{animal["image_url"].iloc[0]}" width="200" height="200" style="filter: sepia(100%) saturate(300%) hue-rotate(-50deg);" />'))
            print("Enter a new animal to play again...")
    else:  
        get_event()
        show_event()


#display the choice buttons and scenatio image
def show_event():
    # print the initial pre-round text layout
    with output: 

        #clear the prev output 
        output.clear_output(wait=True)

        update_stats_display()

        #add strings to widget (as HTML for a pretty layout)
        event_title = widgets.HTML(f"<h3>{curr_event['name']}</h3>")
        event_desc = widgets.HTML(f"<p>{curr_event['description']}</p>")


        # make buttons, if a button is clicked, apply the impact of the choice made
        buttons = []
        for choice in choices.values():
            button = widgets.Button(
                description=choice["description"], 
                layout=widgets.Layout(width='95%', margin='2px')
            )
            button.on_click(lambda b, c=choice: choice_selected(b, c))
            buttons.append(button)


        #add picture of animal & scenario to widgit
        animal_img = widgets.HTML(f'<img src="{animal["image_url"].iloc[0]}" width="180" height="180" style="border-radius: 8px;" />')
        scenario_img = widgets.HTML(f'<img src="{curr_event["scenario_image"]}" width="180" height="180" style="border-radius: 8px;" />')

        
        # create a grid display FIGURE THIS OUTTTT
        
        # 5. ASSEMBLE THE COLUMNS FIRST
        # Create the vertical stats stack
        stats_column = widgets.VBox([widgets.Label(value=f"Animal: {animal['animal'].iloc[0]}"), hp_bar, nrg_bar, score_label])
        
        # Create the vertical gameplay choice stack
        gameplay_column = widgets.VBox([event_title, event_desc, widgets.VBox(buttons)], layout=widgets.Layout(width='60%'))
        
        # 6. ASSEMBLE THE ROWS SECOND (Using the columns we just created above)
        # Row 1 (Top): Put the Animal Pic and the Stats Column side-by-side
        top_row = widgets.HBox([animal_img, stats_column], layout=widgets.Layout(align_items='center', margin='0 0 15px 0'))
        
        # Row 2 (Bottom): Put the Text/Buttons and the Scenario Pic side-by-side
        bottom_row = widgets.HBox([gameplay_column, scenario_img], layout=widgets.Layout(align_items='flex-start'))
        
        # 7. MASTER GAME FRAME
        game_board = widgets.VBox([top_row, widgets.HTML("<hr style='border: 0.5px solid #ccc;'/>"), bottom_row])
        
        # Render the final layout to the screen all at once
        display(game_board)
        


In [349]:
#show image from url
def display_url(url):
    display(HTMLImage(url=url, width=200, height=200))

#I discovered that this was denied by the webiste that has the images... pivoted to HTML display to accomodate
# def display_url(url):
#     response = requests.get(url)
#     img = widgets.Image(
#         value=response.content,
#         format="jpeg",
#         width=200,
#         height=200
#     )
#     display(img)


In [339]:
def start_game(animal):
    global animal_hp, animal_nrg, player_score

    #reset the stats
    animal_hp, animal_nrg = set_stats(animal)
    player_score = 

    # map the starting stats to the display bars
    hp_bar.max = animal_hp if animal_hp > 100 else 100
    nrg_bar.max = animal_nrg if animal_nrg > 100 else 100


    #init the scenarios 
    get_event()
    show_event()

In [308]:
#CHAT DO NOT WORRY ABOUT THIS YET
#visualize the stats better (meter w number?) can I do things side by side? 
#Allow the clicking to not move the page so it feels cohesive

In [381]:
#THIS CELL RUNS THE GAME!!!!!!!!!
output = widgets.Output()

text_input = widgets.Text(placeholder = 'Enter an animal')
text_input.observe(select_animal, names="value")
#text_input.continuous_update = False

display(text_input)
display(output)

Text(value='', placeholder='Enter an animal')

Output()

In [380]:
#create the stat display components
hp_bar = widgets.IntProgress(value=30, min=0, max=100, description='Health: ', bar_style='danger')
nrg_bar = widgets.IntProgress(value=30, min=0, max=100, description='Energy:', bar_style='warning')
score_label = widgets.Label(value="Score: 0")

def update_stats_display():
    hp_bar.value = max(0, animal_hp)
    nrg_bar.value = max(0, animal_nrg)
    
    # Update the text on our score label
    score_label.value = f"Score: {player_score}"

In [ ]:
#figure out the columns and whatnot
#add the health saats as numbers
